## 1. Instalación de Dependencias

Usamos el stack oficial de LangChain con el conector de Google Gemini (`langchain-google-genai`) para el LLM y los embeddings, `langchain-chroma` para la base vectorial y `pillow` para generar un recibo de prueba.

In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-chroma chromadb pillow pandas ipywidgets

print("Instalacion completa.")

Instalacion completa.


#### 1.1 Instalacion de GRADIO para creacion de chat


In [ ]:
%pip install -U \
transformers==4.44.2 \
tokenizers==0.19.1 \
huggingface-hub==0.24.7 \
gradio
print("Instalacion completa.")

## 1.2 Conexion a Google Gemini y variables de entorno

Configuramos **un mismo proveedor (Gemini)** para todo:

- **LLM:** `gemini-2.0-flash` — es **multimodal**, asi que sirve tanto para texto como para leer imagenes.
- **Embeddings:** `models/text-embedding-004` — convierte el texto de la politica en vectores.

### 🔑 API Key
1. Ve a [Google AI Studio](https://aistudio.google.com/apikey)
2. Inicia sesion y haz clic en **Create API Key**
3. Copia la clave y pegala abajo (o usa `getpass` para no dejarla escrita).

In [ ]:
import os, getpass
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Se solicita la clave de forma segura (no queda escrita en el notebook)
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")

MODELO_LLM = "gemini-2.0-flash"
MODELO_EMBEDDING = "models/text-embedding-004"

llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

# Ping: si esto imprime el saludo, estas conectado
print(llm.invoke("Responde unicamente: 'Gemini conectado.' y nada mas.").content)


## 2. Carga documental de las bases de conocimiento de los Agentes (BENEFICIOS_COMPENSACIONES||REGLAMENTO_INTERNO||RECLUTAMIENTO_ONBOARDING)

**Descripción:**
El agente de conocimiento no debe **inventar** las reglas: debe responder a partir de los documentos oficiales. La celda siguiente crea el archivo lee los archivos de la carpeta 5_RRHH y lo carga.

In [ ]:
from pathlib import Path

# Carpeta donde deben estar los documentos de RRHH
carpeta = Path("5_RRHH")

if not carpeta.exists():
    print(f"⚠️ La carpeta '{carpeta}' no existe todavia.")
    print("Crea la carpeta y coloca dentro los archivos:")
    print(" - 01_Beneficios_Compensaciones.txt")
    print(" - 02_Reglamento_Interno.txt")
    print(" - 03_Reclutamiento_Onboarding.txt")
else:
    documentos = []
    for archivo in carpeta.glob("*.txt"):
        contenido = archivo.read_text(encoding="utf-8")
        documentos.append(contenido)
        print(f"{archivo.name}: {len(contenido)} caracteres")

    if not documentos:
        print("⚠️ No se encontraron archivos .txt en la carpeta '5_RRHH'.")


## 3. Chunking, Embeddings y Bases Vectoriales

**Descripción:**
En esta etapa se implementa el pipeline **RAG (Retrieval-Augmented Generation)** para cada agente especializado. Los documentos son segmentados en fragmentos (*chunks*), transformados en representaciones vectoriales mediante **GoogleGenerativeAIEmbeddings** y almacenados en índices independientes de **Chroma**, garantizando que cada agente consulte únicamente su propia base de conocimiento.

Durante la recuperación se emplea un **top_k = 3**, seleccionando los tres fragmentos con mayor similitud semántica respecto a la consulta del usuario. Esta estrategia optimiza la precisión de las respuestas, reduce el contexto enviado al modelo y minimiza el riesgo de alucinaciones, asegurando respuestas trazables y fundamentadas en la información documental.


In [ ]:
import re
from pathlib import Path
from langchain_chroma import Chroma

# Carpeta donde están los archivos .txt
BASE_PATH = Path("5_RRHH")  # antes: Path(".") -> por eso no encontraba los documentos

def chunkear_por_seccion(texto):
    """Crea un chunk por cada sección numerada (1., 2., 3., ...)."""
    cabeceras = list(re.finditer(r"^\d+\.\s", texto, flags=re.MULTILINE))
    chunks = []

    for i, m in enumerate(cabeceras):
        inicio = m.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        chunks.append(texto[inicio:fin].strip())

    return chunks

# Listas donde se almacenarán todos los chunks y sus metadatos
todos_los_chunks = []
metadatas = []

# Leer automáticamente todos los archivos TXT
for archivo in BASE_PATH.glob("*.txt"):

    print(f"Procesando: {archivo.name}")

    texto = archivo.read_text(encoding="utf-8")

    chunks = chunkear_por_seccion(texto)

    print(f"  Chunks encontrados: {len(chunks)}")

    for i, chunk in enumerate(chunks):

        todos_los_chunks.append(chunk)

        metadatas.append({
            "fuente": archivo.name,
            "seccion": i + 1
        })

print("\n===================================")
print(f"Total de chunks: {len(todos_los_chunks)}")
print("===================================")

if not todos_los_chunks:
    raise ValueError(
        "No se encontraron chunks. Verifica que la carpeta '5_RRHH' exista junto al notebook y contenga los archivos "
        "01_Beneficios_Compensaciones.txt, 02_Reglamento_Interno.txt "
        "y 03_Reclutamiento_Onboarding.txt con secciones numeradas (1., 2., 3. ...)."
    )

# Crear la base vectorial en Chroma utilizando los embeddings de Gemini
vectorstore = Chroma.from_texts(
    texts=todos_los_chunks,
    embedding=embeddings,
    metadatas=metadatas,
    collection_name="rrhh_patito"
)

# Crear el retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("\n✅ Base de conocimiento creada correctamente.")

## 4. Agentes RAG

## 4.1. Agente de Beneficios y compensaciones

**Descripción:**
Agente especializado en responder consultas relacionadas con los beneficios corporativos, compensaciones y programas de bienestar para los colaboradores. Utiliza un sistema RAG para recuperar información desde la documentación institucional y proporcionar respuestas precisas, contextualizadas y fundamentadas en las políticas vigentes, evitando generar información que no se encuentre en la base de conocimiento.

In [ ]:
PROMPT_BENEFICIOS = """
Eres el Agente de Beneficios y Compensaciones de PATITO S.A.

Especialidad:
- Seguro médico
- Dependientes
- Bonificaciones
- Beneficios laborales
- Compensaciones

Responde únicamente utilizando la información recuperada del documento
'01_Beneficios_Compensaciones.txt'.

Si la respuesta no aparece en el contexto proporcionado, responde:

"No encontré información sobre ese tema en el documento de Beneficios y Compensaciones."

No inventes información.
Responde de forma clara, profesional y precisa.
"""

def responder_beneficios(pregunta: str) -> str:

    # Solo consulta el documento de Beneficios
    docs = vectorstore.similarity_search(
        pregunta,
        k=3,
        filter={"fuente": "01_Beneficios_Compensaciones.txt"}
    )

    contexto = "\n\n---\n\n".join(d.page_content for d in docs)

    msg = llm.invoke([
        {"role": "system", "content": PROMPT_BENEFICIOS},
        {
            "role": "user",
            "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}"
        },
    ])

    return msg.content
    #prueba
print(responder_beneficios("¿Qué beneficios ofrece la empresa?"))   

## 4.2. Agente de Políticas Internas

**Descripción:**
Agente especializado en responder consultas relacionadas con las políticas, reglamentos y normativas internas de la organización. Emplea un sistema RAG para recuperar información desde la documentación oficial de la empresa y generar respuestas precisas, contextualizadas y alineadas con las disposiciones vigentes, evitando proporcionar información que no esté respaldada por la base de conocimiento.

In [ ]:
PROMPT_REGLAMENTO = """
Eres el Agente de Reglamento Interno de PATITO S.A.

Especialidad:
- Jornada laboral
- Vacaciones
- Permisos
- Código de conducta
- Sanciones disciplinarias

Responde únicamente utilizando la información recuperada del documento
'02_Reglamento_Interno.txt'.

Si la respuesta no aparece en el contexto proporcionado, responde:

"No encontré información sobre ese tema en el Reglamento Interno de PATITO S.A."

No inventes información.
Responde de forma clara, profesional y precisa.
"""

def responder_reglamento(pregunta: str) -> str:

    # Recuperar únicamente información del Reglamento Interno
    docs = vectorstore.similarity_search(
        pregunta,
        k=3,
        filter={"fuente": "02_Reglamento_Interno.txt"}
    )

    contexto = "\n\n---\n\n".join(d.page_content for d in docs)

    msg = llm.invoke([
        {
            "role": "system",
            "content": PROMPT_REGLAMENTO
        },
        {
            "role": "user",
            "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}"
        }
    ])

    return msg.content
 #prueba
print(responder_reglamento("¿Cuántos días de vacaciones tengo?"))  

## 4.3. Agente de Reclutamiento y Onboarding

**Descripción:**
Este agente especializado implementa el patrón RAG (Retrieval-Augmented Generation) para responder consultas relacionadas con procesos de reclutamiento, selección, programa de referidos e incorporación de nuevos colaboradores (Onboarding). Su funcionamiento se basa en una base de conocimiento independiente, compuesta por el documento Guía de Reclutamiento, Referidos y Onboarding, previamente procesado mediante chunking, embebido con GoogleGenerativeAIEmbeddings e indexado en Chroma.

Durante la recuperación se emplea un top_k = 3, seleccionando los fragmentos más relevantes para generar respuestas precisas, trazables y fundamentadas exclusivamente en la documentación del dominio, evitando la generación de información no respaldada por la base de conocimiento.

In [ ]:
PROMPT_RECLUTAMIENTO = """
Eres el Agente de Reclutamiento y Onboarding de PATITO S.A.

Especialidad:
- Proceso de selección
- Programa de referidos
- Documentación de ingreso
- Onboarding
- Inducción
- Plan 30-60-90

Responde únicamente utilizando la información recuperada del documento
'03_Reclutamiento_Onboarding.txt'.

Si la respuesta no aparece en el contexto proporcionado, responde:

"No encontré información sobre ese tema en el documento de Reclutamiento y Onboarding de PATITO S.A."

No inventes información.
Responde de forma clara, profesional y precisa.
"""

def responder_reclutamiento(pregunta: str) -> str:

    # Recuperar únicamente información del documento de Reclutamiento y Onboarding
    docs = vectorstore.similarity_search(
        pregunta,
        k=3,
        filter={"fuente": "03_Reclutamiento_Onboarding.txt"}
    )

    contexto = "\n\n---\n\n".join(d.page_content for d in docs)

    msg = llm.invoke([
        {
            "role": "system",
            "content": PROMPT_RECLUTAMIENTO
        },
        {
            "role": "user",
            "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}"
        }
    ])

    return msg.content
     #prueba
print(responder_reclutamiento("¿Qué documentos debo presentar para ingresar a la empresa?")) 

## 5. Agente Multimodal de Imagen

**Descripción:**
Agente encargado de analizar imágenes y documentos visuales, extraer información relevante e identificar elementos clave para generar respuestas claras y contextualizadas dentro de los procesos de Recursos Humanos.


In [ ]:
from langchain_core.messages import HumanMessage
from pathlib import Path
import base64
import mimetypes
import json
import re

PROMPT_MULTIMODAL = """
Eres el Agente Multimodal del Departamento de Recursos Humanos de PATITO S.A.

Tu función es analizar documentos laborales enviados como imagen.

Analiza cuidadosamente la imagen y responde únicamente con la información visible.

Extrae:

- Tipo de documento.
- Nombre del colaborador.
- Cargo.
- Departamento.
- Número o ID del empleado.
- Fecha de ingreso.
- Correo electrónico (si existe).
- Cualquier otra información relevante.

Si algún dato no aparece responde:

"No visible"

Finalmente indica:

Estado del documento:
- Legible
- Parcialmente legible
- No legible

No inventes información.
"""


def analizar_documento_rrhh(
    ruta_imagen: str,
    devolver_json=False,
    rol_documento="COLABORADOR"
):

    archivo = Path(ruta_imagen)

    if not archivo.exists():

        if devolver_json:
            return {
                "rol_documento": rol_documento,
                "nombre": None,
                "cedula": None,
                "fecha_nacimiento": None
            }

        return f"No se encontró la imagen: {ruta_imagen}"


    mime_type, _ = mimetypes.guess_type(ruta_imagen)

    formatos = [
        "image/png",
        "image/jpeg",
        "image/webp"
    ]


    if mime_type not in formatos:

        if devolver_json:
            return {
                "rol_documento": rol_documento,
                "nombre": None,
                "cedula": None,
                "fecha_nacimiento": None
            }

        return (
            "Formato de imagen no soportado.\n"
            "Utilice PNG, JPG, JPEG o WEBP."
        )


    with open(ruta_imagen, "rb") as archivo_imagen:

        imagen_b64 = base64.b64encode(
            archivo_imagen.read()
        ).decode()


    # -----------------------------------------------------
    # Modo normal: describir el documento
    # -----------------------------------------------------

    if not devolver_json:

        mensaje = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": PROMPT_MULTIMODAL
                },
                {
                    "type": "image_url",
                    "image_url": (
                        f"data:{mime_type};base64,{imagen_b64}"
                    )
                }
            ]
        )

        respuesta = llm.invoke(
            [mensaje]
        )

        return respuesta.content


    # -----------------------------------------------------
    # Modo JSON: extraer datos para una solicitud
    # -----------------------------------------------------

    prompt_json = f"""
Analiza cuidadosamente el documento de identidad adjunto.

El documento pertenece al rol:

{rol_documento}

Extrae únicamente los datos que aparezcan claramente en la imagen.

Devuelve exclusivamente un JSON válido con este formato:

{{
    "rol_documento": "{rol_documento}",
    "nombre": null,
    "cedula": null,
    "fecha_nacimiento": null
}}

REGLAS:

- En "nombre" coloca los nombres y apellidos completos.
- En "cedula" coloca únicamente el número de identificación.
- En "fecha_nacimiento" conserva la fecha que aparece en el documento.
- Si un dato no aparece o no es legible, coloca null.
- No inventes información.
- No agregues explicaciones.
- No uses bloques Markdown.
"""


    mensaje = HumanMessage(
        content=[
            {
                "type": "text",
                "text": prompt_json
            },
            {
                "type": "image_url",
                "image_url": (
                    f"data:{mime_type};base64,{imagen_b64}"
                )
            }
        ]
    )


    respuesta = llm.invoke(
        [mensaje]
    )

    contenido = respuesta.content


    if isinstance(contenido, list):

        texto = ""

        for bloque in contenido:

            if (
                isinstance(bloque, dict)
                and bloque.get("type") == "text"
            ):

                texto += bloque.get(
                    "text",
                    ""
                )

            else:

                texto += str(bloque)

    else:

        texto = str(contenido)


    texto = texto.strip()

    texto = re.sub(
        r"^```json",
        "",
        texto,
        flags=re.IGNORECASE
    ).strip()

    texto = re.sub(
        r"^```",
        "",
        texto
    ).strip()

    texto = re.sub(
        r"```$",
        "",
        texto
    ).strip()


    try:

        datos = json.loads(
            texto
        )

        if not isinstance(datos, dict):
            raise ValueError(
                "La respuesta no es un objeto JSON."
            )


        return {
            "rol_documento": datos.get(
                "rol_documento",
                rol_documento
            ),
            "nombre": datos.get("nombre"),
            "cedula": datos.get("cedula"),
            "fecha_nacimiento": datos.get(
                "fecha_nacimiento"
            )
        }


    except Exception as error:

        print(
            "Error interpretando imagen:",
            error
        )

        print(
            "Respuesta multimodal:",
            texto
        )

        return {
            "rol_documento": rol_documento,
            "nombre": None,
            "cedula": None,
            "fecha_nacimiento": None
        }

### 6. Agente de Acción para solicitudes de RR. HH.

**Descripción:**
Agente encargado de ejecutar y gestionar solicitudes operativas de Recursos Humanos, como el registro de vacaciones, permisos, certificados y postulaciones internas. Valida los datos proporcionados por el colaborador, solicita confirmación antes de realizar la acción y comunica el resultado de la gestión.


In [ ]:
import json

PROMPT_EXTRACCION = """
Eres un asistente de Recursos Humanos de PATITO S.A.

Analiza la solicitud del usuario y extrae los trámites que desea realizar.

Una solicitud puede contener uno o varios trámites.

Los tipos permitidos son:

- VACACIONES
- DEPENDIENTE


Responde únicamente con un JSON válido.


Ejemplo 1:

Usuario:
"Quiero solicitar vacaciones"

Respuesta:

{
    "solicitudes": [
        {
            "tipo":"VACACIONES",
            "nombre":"",
            "fecha_inicio":"",
            "fecha_fin":"",
            "dias":"",
            "jefe":""
        }
    ]
}


Ejemplo 2:

Usuario:
"Quiero agregar a mi pareja al seguro médico"

Respuesta:

{
    "solicitudes": [
        {
            "tipo":"DEPENDIENTE",
            "colaborador":"",
            "dependiente":"",
            "vinculo":"PAREJA",
            "documentos":""
        }
    ]
}


Ejemplo 3:

Usuario:
"Quiero tomar vacaciones y agregar a mi pareja al seguro"

Respuesta:

{
    "solicitudes": [
        {
            "tipo":"VACACIONES",
            "nombre":"",
            "fecha_inicio":"",
            "fecha_fin":"",
            "dias":"",
            "jefe":""
        },
        {
            "tipo":"DEPENDIENTE",
            "colaborador":"",
            "dependiente":"",
            "vinculo":"PAREJA",
            "documentos":""
        }
    ]
}


Reglas:

- Si un dato no aparece, déjalo vacío.
- No inventes información.
- No agregues explicaciones.
- Devuelve únicamente JSON válido.
"""

## 7. FUNCIONES

### 7.1. Extracción estructurada de datos
**Descripción:**
Proceso mediante el cual la información relevante extraída de documentos o imágenes es identificada, organizada y transformada en datos estructurados. Esto permite validar la información, automatizar procesos de Recursos Humanos y facilitar su posterior almacenamiento y procesamiento por el sistema.

In [ ]:
def extraer_datos(solicitud: str, contexto=None):

    contexto_texto = ""

    if contexto:

        contexto_texto = f"""
Contexto de solicitud pendiente:

{json.dumps(
    contexto,
    ensure_ascii=False,
    indent=2
)}
"""


    respuesta = llm.invoke([
        {
            "role": "system",
            "content": PROMPT_EXTRACCION
        },
        {
            "role": "user",
            "content": f"""
{contexto_texto}

Nueva información del usuario:

{solicitud}

Extrae únicamente la información nueva.
"""
        }
    ])


    contenido = respuesta.content


    if isinstance(contenido, list):

        texto = ""

        for bloque in contenido:

            if isinstance(bloque, dict) and bloque.get("type") == "text":

                texto += bloque.get("text","")

    else:

        texto = str(contenido)



    texto = texto.strip()


    texto = re.sub(
        r"^```json",
        "",
        texto,
        flags=re.IGNORECASE
    ).strip()


    texto = re.sub(
        r"^```|```$",
        "",
        texto
    ).strip()


    try:

        return json.loads(texto)


    except:

        print("JSON inválido:")
        print(texto)

        return None

## 7.2. Validación, confirmación y registro

**Descripción:**
Proceso encargado de verificar la integridad y consistencia de los datos proporcionados por el colaborador, solicitar una confirmación antes de ejecutar la solicitud y registrar la información una vez validada, garantizando la confiabilidad y trazabilidad de las operaciones realizadas por el sistema.

In [ ]:
def validar_datos(datos):

    if datos is None:
        return ["No fue posible interpretar la solicitud."]


    if "solicitudes" not in datos:

        return ["No se encontraron solicitudes válidas."]


    faltantes = []


    for solicitud in datos["solicitudes"]:


        tipo = solicitud.get("tipo")



        if tipo == "VACACIONES":

            obligatorios = [
                "nombre",
                "fecha_inicio",
                "fecha_fin",
                "dias",
                "jefe"
            ]



        elif tipo == "DEPENDIENTE":

            obligatorios = [
                "colaborador",
                "dependiente",
                "vinculo"
            ]



        else:

            faltantes.append(
                "Tipo de solicitud desconocido."
            )

            continue



        for campo in obligatorios:


            valor = solicitud.get(campo)


            if valor is None or str(valor).strip() == "":

                faltantes.append(
                    f"{tipo}: {campo}"
                )



    return faltantes

### 7.3. Función completar_datos()
**Descripción:**
Función encargada de identificar y completar la información requerida para procesar una solicitud de Recursos Humanos. Si detecta datos faltantes, interactúa con el colaborador para solicitarlos y consolidar un conjunto de datos completo antes de continuar con la validación y ejecución de la operación.

In [ ]:
def completar_datos(datos_actuales, nuevos_datos):

    if not datos_actuales:

        return nuevos_datos


    if not isinstance(datos_actuales, dict):

        return datos_actuales


    solicitudes = datos_actuales.get(
        "solicitudes",
        []
    )


    if not solicitudes:

        return datos_actuales


    if not nuevos_datos:

        return datos_actuales


    # -----------------------------------------------------
    # Datos provenientes de una imagen
    # -----------------------------------------------------

    if (
        isinstance(nuevos_datos, dict)
        and "rol_documento" in nuevos_datos
    ):

        rol = str(
            nuevos_datos.get(
                "rol_documento",
                "COLABORADOR"
            )
        ).upper().strip()

        nombre = nuevos_datos.get("nombre")
        cedula = nuevos_datos.get("cedula")

        fecha_nacimiento = nuevos_datos.get(
            "fecha_nacimiento"
        )


        for solicitud in solicitudes:

            tipo = solicitud.get("tipo")


            # -------------------------------------------------
            # Documento del colaborador
            # -------------------------------------------------

            if rol == "COLABORADOR":

                if tipo == "VACACIONES":

                    if nombre and not solicitud.get("nombre"):

                        solicitud["nombre"] = nombre


                    if cedula and not solicitud.get("cedula"):

                        solicitud["cedula"] = cedula


                elif tipo == "DEPENDIENTE":

                    if (
                        nombre
                        and not solicitud.get("colaborador")
                    ):

                        solicitud["colaborador"] = nombre


                    if (
                        cedula
                        and not solicitud.get(
                            "cedula_colaborador"
                        )
                    ):

                        solicitud[
                            "cedula_colaborador"
                        ] = cedula


            # -------------------------------------------------
            # Documento del dependiente
            # -------------------------------------------------

            elif rol == "DEPENDIENTE":

                if tipo == "DEPENDIENTE":

                    if (
                        nombre
                        and not solicitud.get("dependiente")
                    ):

                        solicitud["dependiente"] = nombre


                    if (
                        cedula
                        and not solicitud.get(
                            "cedula_dependiente"
                        )
                    ):

                        solicitud[
                            "cedula_dependiente"
                        ] = cedula


                    if (
                        fecha_nacimiento
                        and not solicitud.get(
                            "fecha_nacimiento_dependiente"
                        )
                    ):

                        solicitud[
                            "fecha_nacimiento_dependiente"
                        ] = fecha_nacimiento


        return {
            "solicitudes": solicitudes
        }


    # -----------------------------------------------------
    # Datos provenientes de texto
    # -----------------------------------------------------

    if (
        isinstance(nuevos_datos, dict)
        and "solicitudes" in nuevos_datos
    ):

        nuevos = nuevos_datos.get(
            "solicitudes",
            []
        )

    else:

        nuevos = [
            nuevos_datos
        ]


    for nuevo in nuevos:

        if not isinstance(nuevo, dict):
            continue


        tipo_nuevo = nuevo.get("tipo")


        for solicitud in solicitudes:

            tipo_solicitud = solicitud.get("tipo")


            if (
                tipo_nuevo
                and tipo_solicitud
                and tipo_nuevo != tipo_solicitud
            ):
                continue


            for campo, valor in nuevo.items():

                if campo == "tipo":
                    continue


                if valor in [
                    None,
                    "",
                    [],
                    {}
                ]:
                    continue


                if not solicitud.get(campo):

                    solicitud[campo] = valor


    return {
        "solicitudes": solicitudes
    }

### 7.4. Función ejecutar_accion()

**Descripción:**
Función responsable de ejecutar la operación solicitada por el colaborador una vez que los datos han sido completados, validados y confirmados. Coordina la invocación de la herramienta correspondiente, realiza el registro de la solicitud y devuelve el resultado de la ejecución al usuario

In [ ]:
def ejecutar_accion(solicitud=None, datos=None):

    # Si no vienen los datos completos,
    # extraerlos desde el texto.
    if datos is None:

        datos = extraer_datos(solicitud)


    if datos is None:

        return "❌ No fue posible interpretar la solicitud."


    faltantes = validar_datos(datos)


    if faltantes:

        mensaje = (
            "❌ No es posible registrar la solicitud.\n\n"
            "Faltan los siguientes datos:\n\n"
        )

        for campo in faltantes:

            mensaje += f"• {campo}\n"

        return mensaje


    resumen = """
Resumen de solicitudes

"""


    for item in datos.get("solicitudes", []):


        tipo = item.get("tipo")


        if tipo == "VACACIONES":

            resumen += f"""
----------------------------

Tipo:
Vacaciones

Nombre:
{item.get('nombre')}

Fecha inicio:
{item.get('fecha_inicio')}

Fecha fin:
{item.get('fecha_fin')}

Número de días:
{item.get('dias')}

Jefe:
{item.get('jefe')}

"""


        elif tipo == "DEPENDIENTE":

            resumen += f"""
----------------------------

Tipo:
Inscripción de dependiente

Colaborador:
{item.get('colaborador')}

Dependiente:
{item.get('dependiente')}

Vínculo:
{item.get('vinculo')}

"""


    resumen += """

----------------------------

¿Confirma registrar estas solicitudes? (SI/NO)

"""

    return resumen

### 7.5. Función confirmar_registro()

**Descripción:**
Función encargada de gestionar la confirmación final del colaborador antes de ejecutar el registro de una solicitud. Tras recibir la aprobación, invoca el proceso correspondiente para completar la operación; en caso de rechazo, cancela la solicitud y notifica el resultado, garantizando que ninguna acción se realice sin la autorización del usuario.

In [ ]:
def confirmar_registro(confirmacion, datos):

    if confirmacion.strip().upper() != "SI":

        return "❌ Registro cancelado."


    if datos is None or "solicitudes" not in datos:

        return "❌ No existen datos para registrar."


    resultados = []


    for solicitud in datos["solicitudes"]:


        tipo = solicitud.get("tipo")


        if tipo == "VACACIONES":


            detalle = f"""
Fecha inicio: {solicitud.get('fecha_inicio')}
Fecha fin: {solicitud.get('fecha_fin')}
Número de días: {solicitud.get('dias')}
Jefe que aprueba: {solicitud.get('jefe')}
"""


            resultado = registrar_solicitud.invoke(
                {
                    "tipo": "Vacaciones",
                    "nombre": solicitud.get("nombre"),
                    "detalle": detalle
                }
            )


            resultados.append(resultado)



        elif tipo == "DEPENDIENTE":


            detalle = f"""
Dependiente: {solicitud.get('dependiente')}
Vínculo: {solicitud.get('vinculo')}
Documentos: {solicitud.get('documentos')}
"""


            resultado = registrar_solicitud.invoke(
                {
                    "tipo": "Inscripción de dependiente",
                    "nombre": solicitud.get("colaborador"),
                    "detalle": detalle
                }
            )


            resultados.append(resultado)



        else:


            resultados.append(
                f"❌ Tipo de solicitud desconocido: {tipo}"
            )



    return (
        "✅ Solicitudes procesadas correctamente:\n\n"
        +
        "\n\n".join(resultados)
    )

### 7.6. Herramienta LangChain de registro||Funcion registrar_solicitud()

**Descripción:**
Componente desarrollado con LangChain que permite automatizar el procesamiento y registro de solicitudes de Recursos Humanos. Se encarga de recibir los datos validados, ejecutar la acción correspondiente y gestionar la interacción entre el modelo de lenguaje y las funciones del sistema de forma segura y estructurada

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import uuid


ARCHIVO_REGISTRO = "registro_solicitudes_rrhh.txt"


@tool
def registrar_solicitud(tipo: str, nombre: str, detalle: str) -> str:
    """
    Registra una solicitud individual de RRHH.
    Se ejecuta únicamente después de la confirmación del usuario.
    """

    fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    identificador = (
        f"SOL-{uuid.uuid4().hex[:8].upper()}"
    )


    registro = f"""
==================================================
ID: {identificador}

Fecha registro:
{fecha}

Tipo de solicitud:
{tipo}

Colaborador:
{nombre}

Detalle:
{detalle}
==================================================

"""


    try:

        with open(
            ARCHIVO_REGISTRO,
            "a",
            encoding="utf-8"
        ) as archivo:

            archivo.write(registro)


        return (
            f"✅ Solicitud registrada correctamente.\n"
            f"ID generado: {identificador}\n"
            f"Tipo: {tipo}"
        )


    except Exception as e:


        return (
            "❌ Error al registrar la solicitud: "
            f"{str(e)}"
        )

### 8.Promt Supervisor IA y clasificación de intenciones

**Descripción:**
Componente encargado de analizar la consulta del colaborador, identificar una o varias intenciones y dirigir cada solicitud hacia el agente especializado correspondiente. Coordina la interacción entre los agentes de Beneficios, Políticas Internas, Reclutamiento, Acción y Multimodal, permitiendo consolidar respuestas cuando una consulta requiere la participación de varios módulos.

In [ ]:
PROMPT_SUPERVISOR = """
Eres el Supervisor Inteligente del Departamento de Recursos Humanos de PATITO S.A.

Tu función NO es responder directamente la consulta del usuario.

Tu única función es analizar la intención completa del mensaje y seleccionar
TODOS los agentes especializados que sean necesarios.

Debes responder exclusivamente con una lista JSON de agentes.

Formato obligatorio:

["AGENTE"]

o, cuando existan varias intenciones:

["AGENTE_1", "AGENTE_2", "AGENTE_3"]


AGENTES DISPONIBLES:

BENEFICIOS
REGLAMENTO
RECLUTAMIENTO
ACCION
MULTIMODAL


CRITERIOS DE CLASIFICACIÓN:


BENEFICIOS

Selecciona BENEFICIOS cuando el usuario solicite información sobre beneficios
ofrecidos por la empresa, incluyendo:

• Seguro médico
• Inclusión de dependientes
• Cobertura médica
• Día libre de cumpleaños
• Capacitación
• Apoyo educativo
• Bonos
• Beneficios corporativos
• Compensaciones adicionales


REGLAMENTO

Selecciona REGLAMENTO cuando el usuario solicite información sobre normas,
derechos, obligaciones, requisitos o políticas laborales, incluyendo:

• Vacaciones
• Cantidad de días de vacaciones
• Procedimiento para solicitar vacaciones
• Permisos laborales
• Licencias
• Permiso por matrimonio
• Horarios
• Jornada laboral
• Reglamento interno
• Normas
• Sanciones
• Conducta
• Obligaciones del colaborador


RECLUTAMIENTO

Selecciona RECLUTAMIENTO cuando la consulta sea sobre procesos de talento humano:

• Vacantes
• Postulación interna
• Empleo
• Selección
• Entrevistas
• Candidatos
• Contratación


ACCION

Selecciona ACCION únicamente cuando el usuario exprese intención de ejecutar,
crear o registrar una gestión, incluyendo:

• Crear una solicitud
• Solicitar o registrar vacaciones
• Registrar dependientes
• Agregar una pareja o familiar al seguro médico
• Generar trámites
• Actualizar información

IMPORTANTE:

Una consulta puede contener una solicitud informativa y una intención de acción.

En esos casos debes seleccionar TODOS los agentes correspondientes.

Por ejemplo:

• Si pregunta cuántos días de vacaciones tiene y además quiere solicitarlas,
  selecciona REGLAMENTO y ACCION.

• Si pregunta los requisitos del seguro médico y además quiere registrar
  un dependiente, selecciona BENEFICIOS y ACCION.

• Si pregunta sobre vacaciones, seguro médico y además quiere realizar ambos
  trámites, selecciona REGLAMENTO, BENEFICIOS y ACCION.


MULTIMODAL

Selecciona MULTIMODAL cuando el usuario solicite analizar un archivo o documento:

• Imagen
• Documento
• Credencial
• Factura
• Comprobante
• Contrato
• Archivo escaneado


EJEMPLOS:


Usuario:
"¿Qué beneficios tiene la empresa?"

Respuesta:

["BENEFICIOS"]


Usuario:
"¿Cuántos días de vacaciones tengo?"

Respuesta:

["REGLAMENTO"]


Usuario:
"Quiero solicitar vacaciones del 1 al 7 de agosto."

Respuesta:

["ACCION"]


Usuario:
"¿Cuántos días de vacaciones tengo y cómo puedo solicitarlas?"

Respuesta:

["REGLAMENTO"]


Usuario:
"¿Cuántos días de vacaciones tengo y además quiero registrar mi solicitud?"

Respuesta:

["REGLAMENTO", "ACCION"]


Usuario:
"¿Qué necesito para inscribir a mi pareja en el seguro médico?"

Respuesta:

["BENEFICIOS"]


Usuario:
"Quiero agregar a mi pareja al seguro médico."

Respuesta:

["BENEFICIOS", "ACCION"]


Usuario:
"Voy a tomar mis vacaciones y además quiero agregar a mi pareja al seguro
médico. ¿Cuántos días me corresponden, cómo los solicito y qué necesito para
inscribir a un dependiente?"

Respuesta:

["REGLAMENTO", "BENEFICIOS", "ACCION"]


REGLAS ESTRICTAS:

• Analiza todas las intenciones presentes en el mensaje.
• No selecciones solamente la intención principal.
• No agregues explicaciones.
• No uses bloques Markdown.
• No respondas la consulta del usuario.
• Devuelve únicamente una lista JSON válida.
"""

### 8.1 Función clasificar_agentes()

**Descripción:**
Módulo encargado de analizar la consulta del colaborador y determinar el agente o conjunto de agentes más adecuados para atender la solicitud. Mediante técnicas de procesamiento de lenguaje natural, clasifica la intención de la consulta y optimiza el enrutamiento hacia los componentes especializados, garantizando respuestas precisas y eficientes.

In [ ]:
def clasificar_agentes(pregunta):

    respuesta = llm.invoke([
        {
            "role": "system",
            "content": PROMPT_SUPERVISOR
        },
        {
            "role": "user",
            "content": pregunta
        }
    ])

    contenido = respuesta.content

    if isinstance(contenido, list):

        texto = ""

        for bloque in contenido:

            if isinstance(bloque, dict) and bloque.get("type") == "text":
                texto += bloque.get("text", "")

    else:

        texto = str(contenido)


    texto = texto.strip()

    texto = texto.replace("```json", "")
    texto = texto.replace("```", "")
    texto = texto.strip()


    import json

    try:

        agentes = json.loads(texto)

        if isinstance(agentes, list):

            return [
                agente.upper().strip()
                for agente in agentes
            ]

    except:

        pass


    agentes_validos = [
        "BENEFICIOS",
        "REGLAMENTO",
        "RECLUTAMIENTO",
        "ACCION",
        "MULTIMODAL"
    ]


    encontrados = []

    for agente in agentes_validos:

        if agente in texto.upper():

            encontrados.append(agente)


    if encontrados:

        return encontrados


    return ["BENEFICIOS"]

### 8.2 Función integrar_respuestas()

**Descripción:**
Componente encargado de clasificar la intención de la consulta del colaborador y seleccionar automáticamente el agente especializado más adecuado para procesarla. Su función es optimizar el enrutamiento de las solicitudes, asegurando que cada consulta sea atendida por el módulo correspondiente según su temática o tipo de acción.

In [ ]:
def integrar_respuestas(pregunta, respuestas):

    agentes = []

    respuestas_informativas = []

    respuestas_accion = []


    # -----------------------------------------------------
    # Separar respuestas informativas y de acción
    # -----------------------------------------------------

    for item in respuestas:

        agente = item.get("agente")

        respuesta = item.get("respuesta")

        if agente and agente not in agentes:
            agentes.append(agente)

        if not respuesta:
            continue


        if agente == "ACCION":

            respuestas_accion.append(
                str(respuesta)
            )

        else:

            respuestas_informativas.append(
                {
                    "agente": agente,
                    "respuesta": str(respuesta)
                }
            )


    partes_finales = []


    # -----------------------------------------------------
    # Consolidar respuestas informativas
    # -----------------------------------------------------

    if respuestas_informativas:

        contexto_informativo = ""


        for item in respuestas_informativas:

            contexto_informativo += f"""
AGENTE:
{item["agente"]}

RESPUESTA:
{item["respuesta"]}

--------------------------------

"""


        prompt = f"""
Eres el Supervisor IA de Recursos Humanos de PATITO S.A.

Tu función es consolidar respuestas generadas por agentes especializados.

REGLAS IMPORTANTES:

- No inventes información.
- No agregues requisitos nuevos.
- No solicites documentos adicionales si ningún agente los indicó.
- Usa únicamente la información proporcionada por los agentes.
- Si un agente no encontró información, indícalo claramente.
- Mantén un tono profesional de Recursos Humanos.
- Responde primero las preguntas informativas del usuario.
- Organiza la información por temas cuando existan varias consultas.
- No menciones que estás consolidando respuestas de agentes.
- No incluyas todavía el formulario o solicitud de datos del agente ACCION.
- No agregues despedidas.
- No escribas "Atentamente".
- No firmes como Supervisor de Recursos Humanos.
- No cierres la conversación si existe una solicitud que todavía debe continuar.
- Finaliza únicamente la parte informativa, porque después se agregarán
  los datos requeridos para ejecutar la gestión.

Pregunta del usuario:

{pregunta}

Información entregada por los agentes:

{contexto_informativo}

Genera una respuesta final clara, directa y profesional.
"""


        resultado = llm.invoke(prompt)

        contenido = resultado.content


        # -------------------------------------------------
        # Normalizar respuesta del modelo
        # -------------------------------------------------

        if isinstance(contenido, list):

            texto_informativo = ""

            for bloque in contenido:

                if (
                    isinstance(bloque, dict)
                    and bloque.get("type") == "text"
                ):

                    texto_informativo += bloque.get(
                        "text",
                        ""
                    )

                else:

                    texto_informativo += str(bloque)

        else:

            texto_informativo = str(contenido)


        texto_informativo = texto_informativo.strip()


        if texto_informativo:

            partes_finales.append(
                texto_informativo
            )


    # -----------------------------------------------------
    # Agregar respuestas de ACCION sin modificarlas
    # -----------------------------------------------------

    if respuestas_accion:

        texto_accion = "\n\n".join(
            respuesta.strip()
            for respuesta in respuestas_accion
            if respuesta.strip()
        )


        if texto_accion:

            partes_finales.append(
                texto_accion
            )


    # -----------------------------------------------------
    # Construir respuesta final
    # -----------------------------------------------------

    if partes_finales:

        respuesta_final = "\n\n".join(
            partes_finales
        )

    else:

        respuesta_final = (
            "No fue posible generar una respuesta "
            "para la consulta realizada."
        )


    # -----------------------------------------------------
    # Mostrar agentes participantes
    # -----------------------------------------------------

    if agentes:

        respuesta_final += "\n\nAgentes participantes:\n"

        for agente in agentes:

            respuesta_final += f"• {agente}\n"


    return respuesta_final.strip()

### 8.3. Orquestador Función principal `supervisor()`

**Descripción:**
Función principal que actúa como punto de entrada del sistema de atención inteligente. Coordina el flujo completo de procesamiento de las consultas, gestionando la recepción de solicitudes, la clasificación de intenciones, la selección de agentes especializados, la integración de respuestas, el procesamiento de imágenes cuando corresponde y la ejecución de acciones de Recursos Humanos. Además, administra el estado de la conversación para garantizar una interacción continua y coherente con el colaborador.


In [ ]:
def supervisor(pregunta, ruta_imagen=None):

    global estado_chat

    # Evitar errores si Gradio envía None
    if pregunta is None:
        pregunta = ""

    pregunta_limpia = pregunta.strip()
    pregunta_mayuscula = pregunta_limpia.upper()


    # =====================================================
    # FUNCIÓN INTERNA: REINICIAR ESTADO
    # =====================================================

    def reiniciar_estado():

        return {
            "esperando_confirmacion": False,
            "esperando_datos": False,
            "esperando_respuesta_cedula": False,
            "esperando_imagen_cedula": False,
            "datos": None,
            "solicitudes": None
        }


    # =====================================================
    # FUNCIÓN INTERNA: MOSTRAR DATOS FALTANTES
    # =====================================================

    def construir_mensaje_faltantes(
        faltantes,
        encabezado=(
            "Para gestionar tus solicitudes necesito "
            "la siguiente información:\n\n"
        )
    ):

        mensaje = encabezado

        vacaciones = []
        dependiente = []
        generales = []


        for campo in faltantes:

            campo_texto = str(campo)

            if campo_texto.startswith("VACACIONES"):

                vacaciones.append(
                    campo_texto.replace(
                        "VACACIONES: ",
                        ""
                    )
                )

            elif campo_texto.startswith("DEPENDIENTE"):

                dependiente.append(
                    campo_texto.replace(
                        "DEPENDIENTE: ",
                        ""
                    )
                )

            else:

                generales.append(
                    campo_texto
                )


        if vacaciones:

            mensaje += "📌 Vacaciones:\n"

            for campo in vacaciones:
                mensaje += f"• {campo}\n"

            mensaje += "\n"


        if dependiente:

            mensaje += (
                "📌 Seguro médico - Dependiente:\n"
            )

            for campo in dependiente:
                mensaje += f"• {campo}\n"

            mensaje += "\n"


        if generales:

            mensaje += "📌 Información adicional:\n"

            for campo in generales:
                mensaje += f"• {campo}\n"

            mensaje += "\n"


        return mensaje.strip()


    # =====================================================
    # 1. CONFIRMACIÓN FINAL DEL REGISTRO
    # =====================================================

    if estado_chat.get("esperando_confirmacion"):

        if pregunta_mayuscula in [
            "SI",
            "SÍ",
            "CONFIRMO",
            "ACEPTO"
        ]:

            respuesta = confirmar_registro(
                "SI",
                estado_chat.get("datos")
            )

            estado_chat = reiniciar_estado()

            return respuesta


        elif pregunta_mayuscula in [
            "NO",
            "CANCELAR",
            "CANCELO"
        ]:

            estado_chat = reiniciar_estado()

            return "❌ Solicitud cancelada."


        else:

            return (
                "Por favor, responde SI para registrar "
                "las solicitudes o NO para cancelarlas."
            )


    # =====================================================
    # 2. RESPUESTA SOBRE ADJUNTAR LA CÉDULA
    # =====================================================

    if estado_chat.get("esperando_respuesta_cedula"):

        respuestas_si = [
            "SI",
            "SÍ",
            "CLARO",
            "OK",
            "DALE",
            "DE ACUERDO",
            "VOY A ADJUNTARLA",
            "QUIERO ADJUNTARLA"
        ]

        respuestas_no = [
            "NO",
            "NO TENGO",
            "NO PUEDO",
            "PREFIERO ESCRIBIR",
            "PREFIERO INGRESARLOS",
            "MANUAL",
            "MANUALMENTE"
        ]


        if pregunta_mayuscula in respuestas_si:

            estado_chat[
                "esperando_respuesta_cedula"
            ] = False

            estado_chat[
                "esperando_imagen_cedula"
            ] = True

            estado_chat[
                "esperando_datos"
            ] = False

            return (
                "Perfecto. Adjunta una imagen clara de tu cédula "
                "utilizando el botón de carga de imagen.\n\n"
                "Procura que el documento esté completo, "
                "bien iluminado y que la información sea legible."
            )


        elif pregunta_mayuscula in respuestas_no:

            estado_chat[
                "esperando_respuesta_cedula"
            ] = False

            estado_chat[
                "esperando_imagen_cedula"
            ] = False

            estado_chat[
                "esperando_datos"
            ] = True

            faltantes = validar_datos(
                estado_chat.get("datos")
            )

            mensaje = construir_mensaje_faltantes(
                faltantes,
                encabezado=(
                    "De acuerdo. Puedes proporcionarme "
                    "los datos manualmente.\n\n"
                    "Necesito la siguiente información:\n\n"
                )
            )

            mensaje += (
                "\n\nEscribe todos los datos disponibles "
                "en un solo mensaje."
            )

            return mensaje


        else:

            return (
                "Por favor, responde SI si deseas adjuntar "
                "tu cédula o NO si prefieres escribir "
                "los datos manualmente."
            )


    # =====================================================
    # 3. ESPERANDO LA IMAGEN DE LA CÉDULA
    # =====================================================

    if estado_chat.get("esperando_imagen_cedula"):

        # Permitir que el usuario cambie de opinión
        if pregunta_mayuscula in [
            "NO",
            "CANCELAR",
            "PREFIERO ESCRIBIR",
            "MANUAL",
            "MANUALMENTE"
        ]:

            estado_chat[
                "esperando_imagen_cedula"
            ] = False

            estado_chat[
                "esperando_datos"
            ] = True

            faltantes = validar_datos(
                estado_chat.get("datos")
            )

            mensaje = construir_mensaje_faltantes(
                faltantes,
                encabezado=(
                    "De acuerdo. Continuemos de forma manual.\n\n"
                    "Proporciona la siguiente información:\n\n"
                )
            )

            mensaje += (
                "\n\nEscribe los datos restantes "
                "en un solo mensaje."
            )

            return mensaje


        # Todavía no cargó una imagen
        if not ruta_imagen:

            return (
                
                "📄 Aún no has adjuntado la imagen de tu cédula.\n\n"
                "Por favor, selecciónala utilizando el botón de carga de imagen "
                "y luego presiona **Enviar** para que pueda analizarla automáticamente.\n\n"
                "Si prefieres ingresar los datos manualmente, responde **NO**."
            )


        # Analizar la cédula en modo JSON
        nuevos_datos = analizar_documento_rrhh(
            ruta_imagen,
            devolver_json=True,
            rol_documento="COLABORADOR"
        )


        estado_chat["datos"] = completar_datos(
            estado_chat.get("datos"),
            nuevos_datos
        )


        estado_chat[
            "esperando_imagen_cedula"
        ] = False


        faltantes = validar_datos(
            estado_chat.get("datos")
        )


        if faltantes:

            estado_chat[
                "esperando_datos"
            ] = True

            mensaje = construir_mensaje_faltantes(
                faltantes,
                encabezado=(
                    "✅ La cédula fue analizada correctamente.\n\n"
                    "Se tomaron los datos visibles del documento.\n\n"
                    "Para continuar todavía necesito:\n\n"
                )
            )

            mensaje += (
                "\n\nEscribe los datos restantes "
                "en un solo mensaje."
            )

            return mensaje


        estado_chat[
            "esperando_datos"
        ] = False

        estado_chat[
            "esperando_confirmacion"
        ] = True

        return ejecutar_accion(
            datos=estado_chat.get("datos")
        )


    # =====================================================
    # 4. CONTINUACIÓN MANUAL DE UNA SOLICITUD
    # =====================================================

    if estado_chat.get("esperando_datos"):

        # Si adjunta otra imagen mientras completa datos
        if ruta_imagen:

            pregunta_minuscula = (
                pregunta_limpia.lower()
            )

            expresiones_dependiente = [
                "dependiente",
                "pareja",
                "esposa",
                "esposo",
                "conviviente",
                "cónyuge",
                "conyuge",
                "documento de ella",
                "documento de él",
                "cedula de ella",
                "cédula de ella",
                "cedula de mi pareja",
                "cédula de mi pareja"
            ]

            es_dependiente = any(
                expresion in pregunta_minuscula
                for expresion
                in expresiones_dependiente
            )


            if es_dependiente:

                rol_documento = "DEPENDIENTE"

            else:

                rol_documento = "COLABORADOR"


            nuevos_datos = analizar_documento_rrhh(
                ruta_imagen,
                devolver_json=True,
                rol_documento=rol_documento
            )

        else:

            if not pregunta_limpia:

                return (
                    "Escribe los datos solicitados "
                    "o adjunta una imagen."
                )


            nuevos_datos = extraer_datos(
                pregunta_limpia,
                estado_chat.get("datos")
            )


        estado_chat["datos"] = completar_datos(
            estado_chat.get("datos"),
            nuevos_datos
        )


        faltantes = validar_datos(
            estado_chat.get("datos")
        )


        if faltantes:

            mensaje = construir_mensaje_faltantes(
                faltantes,
                encabezado=(
                    "✅ Se procesó la información proporcionada.\n\n"
                    "Para continuar todavía necesito:\n\n"
                )
            )

            mensaje += (
                "\n\nPuedes escribir los datos faltantes "
                "o adjuntar otro documento."
            )

            return mensaje


        estado_chat[
            "esperando_datos"
        ] = False

        estado_chat[
            "esperando_confirmacion"
        ] = True


        return ejecutar_accion(
            datos=estado_chat.get("datos")
        )


    # =====================================================
    # 5. IMAGEN FUERA DE UNA SOLICITUD
    # =====================================================

    if ruta_imagen:

        return analizar_documento_rrhh(
            ruta_imagen
        )


    # =====================================================
    # 6. SUPERVISOR ORQUESTADOR
    # =====================================================

    if not pregunta_limpia:

        return (
            "Escribe una consulta o adjunta una imagen "
            "para comenzar."
        )


    agentes = clasificar_agentes(
        pregunta_limpia
    )


    print("\nAgentes seleccionados:")

    for agente in agentes:
        print(f"• {agente}")


    respuestas = []


    # =====================================================
    # 7. EJECUTAR LOS AGENTES SELECCIONADOS
    # =====================================================

    for agente in agentes:

        # -------------------------------------------------
        # BENEFICIOS
        # -------------------------------------------------

        if agente == "BENEFICIOS":

            respuestas.append(
                {
                    "agente": "BENEFICIOS",
                    "respuesta": responder_beneficios(
                        pregunta_limpia
                    )
                }
            )


        # -------------------------------------------------
        # REGLAMENTO
        # -------------------------------------------------

        elif agente == "REGLAMENTO":

            respuestas.append(
                {
                    "agente": "REGLAMENTO",
                    "respuesta": responder_reglamento(
                        pregunta_limpia
                    )
                }
            )


        # -------------------------------------------------
        # RECLUTAMIENTO
        # -------------------------------------------------

        elif agente == "RECLUTAMIENTO":

            respuestas.append(
                {
                    "agente": "RECLUTAMIENTO",
                    "respuesta": responder_reclutamiento(
                        pregunta_limpia
                    )
                }
            )


        # -------------------------------------------------
        # ACCIÓN
        # -------------------------------------------------

        elif agente == "ACCION":

            datos = extraer_datos(
                pregunta_limpia,
                estado_chat.get("datos")
            )


            if datos is None:

                respuestas.append(
                    {
                        "agente": "ACCION",
                        "respuesta": (
                            "❌ No fue posible interpretar "
                            "la solicitud."
                        )
                    }
                )

                continue


            faltantes = validar_datos(
                datos
            )


            estado_chat["datos"] = datos

            estado_chat["solicitudes"] = datos.get(
                "solicitudes",
                []
            )


            if faltantes:

                estado_chat[
                    "esperando_confirmacion"
                ] = False

                estado_chat[
                    "esperando_datos"
                ] = False

                estado_chat[
                    "esperando_respuesta_cedula"
                ] = True

                estado_chat[
                    "esperando_imagen_cedula"
                ] = False


                mensaje = construir_mensaje_faltantes(
                    faltantes,
                    encabezado=(
                        "Para gestionar tus solicitudes necesito "
                        "completar algunos datos.\n\n"
                    )
                )


                mensaje += (
                    "\n\nPuedes adjuntar una imagen clara "
                    "de tu cédula. Así podré obtener "
                    "automáticamente los datos visibles "
                    "del documento y te resultará más fácil "
                    "completar la solicitud.\n\n"
                    "¿Deseas adjuntar tu cédula? "
                    "Responde SI o NO."
                )


                respuestas.append(
                    {
                        "agente": "ACCION",
                        "respuesta": mensaje
                    }
                )


            else:

                estado_chat[
                    "esperando_confirmacion"
                ] = True

                estado_chat[
                    "esperando_datos"
                ] = False

                estado_chat[
                    "esperando_respuesta_cedula"
                ] = False

                estado_chat[
                    "esperando_imagen_cedula"
                ] = False


                respuestas.append(
                    {
                        "agente": "ACCION",
                        "respuesta": ejecutar_accion(
                            datos=datos
                        )
                    }
                )


        # -------------------------------------------------
        # MULTIMODAL SIN IMAGEN
        # -------------------------------------------------

        elif agente == "MULTIMODAL":

            respuestas.append(
                {
                    "agente": "MULTIMODAL",
                    "respuesta": (
                        "Para realizar el análisis debes "
                        "adjuntar una imagen o documento."
                    )
                }
            )


    # =====================================================
    # 8. CONSOLIDAR RESPUESTA FINAL
    # =====================================================

    if not respuestas:

        return (
            "No fue posible identificar el agente adecuado "
            "para procesar tu consulta."
        )


    return integrar_respuestas(
        pregunta_limpia,
        respuestas
    )

### 8.4. Función responder()||Adaptador responder() para Gradio

**Descripción:**
Función encargada de gestionar la interacción entre el usuario y el sistema mediante la interfaz conversacional. Recibe la consulta del colaborador, el historial de la conversación y, opcionalmente, una imagen o documento, enviando esta información a la función principal supervisor() para su procesamiento. Finalmente, devuelve la respuesta generada y actualiza el historial del chat, permitiendo mantener el contexto de la conversación durante toda la interacción.

In [ ]:
def responder(mensaje, historial, imagen=None):

    if historial is None:
        historial = []

    try:

        if mensaje is None:
            mensaje = ""

        resultado = supervisor(
            mensaje,
            imagen
        )

        # Convertir resultado a texto
        if isinstance(resultado, list):

            partes = []

            for elemento in resultado:

                if isinstance(elemento, dict):

                    texto = elemento.get(
                        "respuesta",
                        str(elemento)
                    )

                    partes.append(str(texto))

                else:

                    partes.append(str(elemento))

            respuesta = "\n\n".join(partes)

        elif isinstance(resultado, dict):

            respuesta = resultado.get(
                "respuesta",
                str(resultado)
            )

        else:

            respuesta = str(resultado)

        historial.append(
            {
                "role": "user",
                "content": mensaje if mensaje else "📎 Documento adjunto"
            }
        )

        historial.append(
            {
                "role": "assistant",
                "content": respuesta
            }
        )

        # Limpiar texto e imagen después del envío
        return "", historial, None

    except Exception as error:

        mensaje_error = (
            "❌ Ocurrió un error al procesar la solicitud.\n\n"
            f"Detalle: {str(error)}"
        )

        historial.append(
            {
                "role": "assistant",
                "content": mensaje_error
            }
        )

        return mensaje, historial, imagen

### 9. Estado Inicial de la conversacion
**Variable de estado estado_chat**

**Descripción:**
Estructura de datos utilizada para administrar el estado de la conversación entre el colaborador y el sistema. Almacena información sobre el flujo actual de la interacción, como la espera de confirmaciones, el ingreso de datos, la recepción de imágenes de cédula y las solicitudes en proceso. Su propósito es mantener el contexto de la conversación, permitiendo que el asistente continúe correctamente cada etapa del proceso sin perder la información previamente proporcionada por el usuario.

In [ ]:
estado_chat = {
    "esperando_confirmacion": False,
    "esperando_datos": False,
    "esperando_respuesta_cedula": False,
    "esperando_imagen_cedula": False,
    "datos": None,
    "solicitudes": None
}

In [ ]:
def nueva_conversacion():
    """Reinicia el estado de la conversacion y limpia el chat."""
    global estado_chat

    estado_chat = {
        "esperando_confirmacion": False,
        "esperando_datos": False,
        "esperando_respuesta_cedula": False,
        "esperando_imagen_cedula": False,
        "datos": None,
        "solicitudes": None
    }

    # outputs=[entrada, chatbot, imagen]
    return "", [], None


CSS_PERSONALIZADO = """
#contenedor-principal {
    max-width: 720px;
    margin: 0 auto;
}
#cabecera {
    text-align: center;
}
#chat-rrhh {
    border-radius: 12px;
}
#entrada-mensaje textarea {
    border-radius: 8px;
}
#pie-pagina {
    text-align: center;
    font-size: 0.8em;
    color: #888888;
    margin-top: 10px;
}
"""


### 10. Interfaz web con Gradio**

**Descripción:**
Interfaz gráfica desarrollada con Gradio que permite al colaborador interactuar con el sistema de Recursos Humanos mediante un chat web. Facilita el envío de consultas, la carga de imágenes o documentos y la visualización de las respuestas generadas por el Supervisor IA, proporcionando una experiencia accesible, intuitiva y centralizada.


In [ ]:
import gradio as gr

print(gr.__version__)

In [39]:
# El CSS ya no se coloca dentro de Blocks en Gradio 6
with gr.Blocks(
    title="Supervisor IA RRHH",
    fill_height=True
) as demo:

    with gr.Column(elem_id="contenedor-principal"):

        gr.Markdown(
            """
            # 🤖 Supervisor IA RRHH

            Consulta beneficios, reglamentos, vacantes y gestiona
            solicitudes de Recursos Humanos.
            """,
            elem_id="cabecera"
        )

        chatbot = gr.Chatbot(
            label="Conversación",
            height=280,
            elem_id="chat-rrhh",
            placeholder=(
                "Hola, soy tu asistente de Recursos Humanos.\n\n"
                "Puedes consultar sobre vacaciones, beneficios, "
                "reglamentos, reclutamiento o registrar solicitudes."
            )
        )

        with gr.Row():

            entrada = gr.Textbox(
                placeholder="Escribe tu consulta o los datos solicitados...",
                lines=1,
                max_lines=3,
                show_label=False,
                scale=8,
                elem_id="entrada-mensaje"
            )

            boton = gr.Button(
                "Enviar ➤",
                variant="primary",
                scale=2,
                elem_id="boton-enviar"
            )

            boton_nueva = gr.Button(
                "🔄 Nueva conversación",
                variant="secondary",
                scale=2,
                elem_id="boton-nueva"
            )

        with gr.Accordion(
            "📎 Adjuntar documento o cédula",
            open=False
        ):

            gr.Markdown(
                """
                Sube una fotografía clara del documento.

                **Recomendaciones:**

                - El documento debe verse completo.
                - Evita reflejos, sombras o imágenes borrosas.
                - Formatos admitidos: JPG, JPEG, PNG y WEBP.
                - 📤 Subir archivo:** selecciona una imagen guardada en tu equipo.
                """
            )

            imagen = gr.Image(
                type="filepath",
                label="Seleccionar imagen",
                sources=["upload"],
                height=200,
                elem_id="carga-documento"
            )

        gr.Markdown(
            "🔒 HECHO POR SQUAD SXK.",
            elem_id="pie-pagina"
        )

    boton.click(
        fn=responder,
        inputs=[entrada, chatbot, imagen],
        outputs=[entrada, chatbot, imagen],
        show_progress="minimal"
    )

    entrada.submit(
        fn=responder,
        inputs=[entrada, chatbot, imagen],
        outputs=[entrada, chatbot, imagen],
        show_progress="minimal"
    )

    boton_nueva.click(
        fn=nueva_conversacion,
        inputs=[],
        outputs=[entrada, chatbot, imagen],
        show_progress="minimal"
    )

demo.queue()

demo.launch(
    inline=True,
    inbrowser=False,
    height=750,
    width="100%",
    css=CSS_PERSONALIZADO
)

* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.
